In [ ]:
pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter

In [12]:
vector_db_path1="d:\\Learnings\\AgenticAI\\MultiAgent\\RAGBOT\\vector_db"
docs_path1="d:\\Learnings\\AgenticAI\\MultiAgent\\RAGBOT\\docs_dir"
collection_name1="document_collection"


In [14]:
def document_ingestion(vector_db_path,docs_path,collection_name):
    loader=DirectoryLoader(
        path=docs_path,
        glob="./*.pdf",
        loader_cls=PyMuPDFLoader
    )
    documents=loader.load()
    text_splitter=CharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=500
    )
    text_chunks=text_splitter.split_documents(documents)
    embedding=HuggingFaceEmbeddings()
    vector_store=Chroma.from_documents(
        documents=text_chunks,
        embedding=embedding,
        persist_directory=vector_db_path,
        collection_name=collection_name
    )
    print("document_ingestion successful")

In [15]:
document_ingestion(vector_db_path1,docs_path1,collection_name1)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5813.86it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


document_ingestion successful


In [23]:
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA

In [24]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [52]:
def retrieval_generaion(vector_db_path,query,collection_name)->list:
    embedding=HuggingFaceEmbeddings()
    llm=ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0
    )
    vector_store=Chroma(
        collection_name=collection_name,
        embedding_function=embedding,
        persist_directory=vector_db_path
    )
    retriever=vector_store.as_retriever()
    qa_chain=RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )
    response=qa_chain.invoke({"query":query})
    return response


In [53]:
query="discuss about Darwin's journey to the Galapagos Islands and his observation ?"
answer=retrieval_generaion(vector_db_path1,query,collection_name1)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8841.62it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [55]:
print(answer["result"])
print("-"*50)
print("*"*50)
print("-"*50)
for source in answer["source_documents"]:
    print(source.metadata)
    print("---"*10)

Charles Darwin's journey to the Galapagos Islands was a pivotal moment in his life that laid the foundation for his groundbreaking theory of evolution through natural selection. In 1831, at the age of 22, Darwin embarked on a five-year voyage aboard the HMS Beagle, a British ship that was tasked with surveying the coast of South America.

The Galapagos Islands, located off the coast of Ecuador, were one of the many stops on the Beagle's journey. Darwin arrived on the islands in September 1835, and he spent about five weeks exploring the islands, collecting specimens, and making observations.

During his time on the Galapagos, Darwin was struck by the unique and diverse wildlife that inhabited the islands. He collected numerous specimens, including finches, tortoises, iguanas, and mockingbirds, which would later become crucial evidence for his theory of evolution.

One of the most significant observations Darwin made on the Galapagos was the variation in the beak shapes and sizes of the